# z-shift -- CO3D teddybear end-to-end rigging demo (Kaggle P100)

Downloads a single CO3D `teddybear` capture (one zip chunk, single-sequence
subset), runs it through the full z-shift chain -- Phase 1 ingest -> Phase 2
MASt3R reconstruction -> Phase 3 mesh refinement -> Phase 5 biped auto-rigging
-- tuned for the best output this data supports, and reports every diagnostic
the pipeline produces along the way.

**Output**: `mesh.glb` (raw), `mesh_refined.glb` (cleaned), `rigged_mesh.glb`
(skinned, Blender-ready), `skeleton.json` + `skinning_weights.json` (rig
metadata), plus a rendered preview and a full JSON diagnostic dump per phase.


## Before you run

- **Settings > Accelerator > GPU P100** (T4 works too, just re-read the timing
  estimates below), **Internet > On** -- required for cloning the repo,
  installing dependencies, downloading the ~2.5 GB MASt3R checkpoint, and
  downloading the CO3D zip.
- Expect roughly **15-30 minutes** end to end. Setup (clone + deps + MASt3R +
  checkpoint download) is the bulk of it; the reconstruction itself is a few
  minutes at 24 frames with exhaustive pairing on a P100.
- CO3D is released under **CC BY-NC 4.0** (non-commercial) by Meta AI
  Research -- fine for a personal/research demo, not for a commercial product.
  Citation: Reizenstein et al., *Common Objects in 3D*, ICCV 2021.


## 1. Imports, paths, config

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

import numpy as np

REPO_URL = "https://github.com/AyushK0808/z-shift.git"
REPO_BRANCH = "main"

# The one CO3D zip chunk this notebook uses -- single-sequence subset,
# teddybear category, chunk 000. ~90 MB on average for this subset (verified
# against the official co3d/links.json manifest), not the ~150-200 GB the
# full (non-single-sequence) teddybear category would cost.
CO3D_ZIP_URL = "https://dl.fbaipublicfiles.com/co3dv2_231130/teddybear_000_singlesequence.zip"

# Frames fed to MASt3R after even subsampling. 24 frames -> C(24,2) = 276
# pairs under `complete` pairing, which at the repo's own measured T4 rate
# (~0.5-0.8 s/pair) is a few minutes of forward passes on a P100 -- affordable
# enough to prefer exhaustive matching over the `swin` windowed approximation
# the pipeline would auto-select above 20 frames. Raise it for denser
# coverage (cost grows ~quadratically); lower it if this is taking too long.
NUM_FRAMES = 24
ARTICULATION = "biped"

ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    OUT_DIR = Path("/kaggle/working/rigging_demo")
    WORK_DIR = Path("/kaggle/temp/rigging_demo")
    REPO_DIR = Path("/kaggle/working/z-shift")
else:
    BASE = Path.cwd() / "rigging_demo_run"
    OUT_DIR = BASE / "out"
    WORK_DIR = BASE / "work"
    REPO_DIR = BASE / "z-shift"

for d in (OUT_DIR, WORK_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("platform :", "Kaggle" if ON_KAGGLE else "other")
print("OUT_DIR  :", OUT_DIR, "(persists -- final outputs go here)")
print("WORK_DIR :", WORK_DIR, "(scratch -- wiped between sessions on Kaggle)")
print("REPO_DIR :", REPO_DIR)


def run(cmd, **kw):
    print("$", " ".join(str(c) for c in cmd))
    subprocess.run(cmd, check=True, **kw)  # noqa: S603 -- fixed, notebook-authored commands

## 2. GPU check (fail fast)

Everything downstream is pointless without a working CUDA GPU, so check
before spending time on the clone/install steps.


In [ ]:
proc = subprocess.run(["nvidia-smi"], capture_output=True, text=True)  # noqa: S607
print(proc.stdout or proc.stderr or "nvidia-smi not found")

import torch

print("torch", torch.__version__, "| cuda build", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA GPU visible to torch. On Kaggle: Settings > Accelerator > "
        "GPU P100 (or T4), Internet > On, then Run All again."
    )
props = torch.cuda.get_device_properties(0)
print(
    f"GPU: {props.name}, {props.total_memory / 1024**3:.1f} GB VRAM, "
    f"compute capability {props.major}.{props.minor}"
)

## 3. Clone z-shift -- no `pip install -e .`

Two real reasons, both confirmed against this project's own `pyproject.toml`
and its existing Kaggle benchmark notebook (`notebooks/tier_b_gpu.ipynb`):

- `pyproject.toml` pins `requires-python >=3.11,<3.12`; Kaggle's Python image
  is newer, so an editable install fails outright.
- `torch`/`torchvision` only get a CUDA-pinned index on Windows
  (`sys_platform == 'win32'`) -- on Linux they resolve from plain PyPI and
  installing them here could replace Kaggle's driver-matched preinstalled
  build with a mismatched one.

Putting `src/` on `sys.path` directly imports `spatial_ingestion` with none
of that risk -- nothing here needs the `zshift-*` console scripts.


In [ ]:
if (REPO_DIR / ".git").exists():
    run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--prune"])
    run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH])
    run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
else:
    run(["git", "clone", "--branch", REPO_BRANCH, "--depth", "50", REPO_URL, str(REPO_DIR)])

for p in (str(REPO_DIR), str(REPO_DIR / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

# PyVista/VTK are used for mesh filtering in Phase 3, never for on-screen
# rendering, but this runtime is headless -- make that explicit.
os.environ["PYVISTA_OFF_SCREEN"] = "true"

if not (REPO_DIR / "src" / "spatial_ingestion").exists():
    raise SystemExit(f"src/spatial_ingestion missing from the clone at {REPO_DIR}")
print("repo ready, spatial_ingestion importable from", REPO_DIR / "src")

## 4. Dependencies

Read straight out of `pyproject.toml`, skipping `torch`/`torchvision` to
keep Kaggle's preinstalled CUDA build, and skipping `numpy` for the same
reason: Kaggle ships numpy 2.x with every other preinstalled package (scipy,
pandas, scikit-learn, torch's own numpy interop) already built against that
ABI. `pyproject.toml` pins `numpy<2` for this project's own torch>=2.4 combo,
but installing that pin here would downgrade numpy in place on top of a
stack still compiled for 2.x -- classic "numpy.dtype size changed, may
indicate binary incompatibility" crash the first time something imports a
not-yet-loaded numpy submodule (e.g. `trimesh` pulling in `numpy.random`).


In [ ]:
try:
    import tomllib
except ModuleNotFoundError:
    import tomli as tomllib  # type: ignore

with open(REPO_DIR / "pyproject.toml", "rb") as fh:
    pyproject = tomllib.load(fh)

SKIP_PREFIXES = ("torch", "torchvision", "numpy")
deps = []
for dep in pyproject["project"]["dependencies"]:
    name = dep.split(">")[0].split("<")[0].split("=")[0].split("[")[0].strip().lower()
    if name.startswith(SKIP_PREFIXES):
        print("skipping", name, "-- keeping Kaggle's preinstalled, ABI-matched build")
        continue
    deps.append(dep)

run([sys.executable, "-m", "pip", "install", "-q", *deps])
print(f"installed {len(deps)} dependencies")

## 5. MASt3R

Mirrors `scripts/setup-mast3r.sh` (same pinned commit) but installs with
`--no-deps`: MASt3R's and DUSt3R's own `requirements.txt` list `torch`, and
letting pip resolve them is the other way to lose the CUDA build. Their real
dependencies are already covered by the project's own dependency list, which
is why `roma`, `einops`, `pyglet<2` and friends were installed above.


In [ ]:
PINNED_MAST3R = "f5209afc300cec36239a7ac992263f36847bbba0"
MAST3R_DIR = REPO_DIR / "third_party" / "mast3r"
DUST3R_DIR = MAST3R_DIR / "dust3r"

PY_STUB = """[build-system]
requires = ["setuptools"]
build-backend = "setuptools.build_meta"

[project]
name = "{name}"
version = "0.1.0"
requires-python = ">=3.10"

[tool.setuptools.packages.find]
where = ["."]
include = ["{name}*"]
"""

if not MAST3R_DIR.exists():
    run(["git", "clone", "https://github.com/naver/mast3r", str(MAST3R_DIR)])
    run(["git", "checkout", PINNED_MAST3R], cwd=MAST3R_DIR)
    run(["git", "submodule", "update", "--init", "--recursive"], cwd=MAST3R_DIR)
else:
    print("mast3r already cloned at", MAST3R_DIR)

for target, name in ((MAST3R_DIR, "mast3r"), (DUST3R_DIR, "dust3r")):
    stub = target / "pyproject.toml"
    if not stub.exists():
        stub.write_text(PY_STUB.format(name=name), encoding="utf-8")
    run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(target)])

# pip install -e registers the package via a .pth/editable-finder entry in
# site-packages, but `site` only scans site-packages for new .pth files at
# interpreter startup -- a kernel already running before this cell executed
# won't see it without a restart. Insert the source dirs directly, same as
# cell 3 already does for z-shift itself, so both packages import in this
# same kernel with no restart required.
for p in (str(MAST3R_DIR), str(DUST3R_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

import dust3r  # noqa: E402,F401
import mast3r  # noqa: E402,F401 -- sanity check the editable install is importable now

print("mast3r importable from", mast3r.__file__)
print("dust3r importable from", dust3r.__file__)

# RoPE CUDA kernels: a speedup, not a requirement. Never fatal -- falls back
# to the plain PyTorch path if the compile fails.
curope = DUST3R_DIR / "croco" / "models" / "curope"
try:
    run([sys.executable, "setup.py", "build_ext", "--inplace"], cwd=curope)
    print("RoPE CUDA kernels compiled")
except subprocess.CalledProcessError:
    print("RoPE kernel compile failed -- not fatal, falling back to the plain PyTorch path")

## 6. Download the CO3D teddybear chunk

Just the one zip chunk (`teddybear_000_singlesequence.zip`), pulled directly
from Meta's CDN using the URL in the official
[`co3d/links.json`](https://github.com/facebookresearch/co3d/blob/main/co3d/links.json)
manifest -- no need to clone the whole `co3d` tool repo or its
`download_dataset.py` machinery for a single file.


In [ ]:
zip_path = WORK_DIR / "teddybear_000_singlesequence.zip"

if not zip_path.exists():
    print("downloading", CO3D_ZIP_URL)
    urllib.request.urlretrieve(CO3D_ZIP_URL, zip_path)
print(f"{zip_path.stat().st_size / 1024**2:.1f} MB")

extract_dir = WORK_DIR / "teddybear_extracted"
if not extract_dir.exists():
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)
print("extracted to", extract_dir)

image_dirs = sorted(p for p in extract_dir.rglob("images") if p.is_dir())
print(f"\nfound {len(image_dirs)} sequence(s):")
for d in image_dirs:
    n = sum(1 for _ in d.iterdir())
    print(" ", d.relative_to(extract_dir), "-", n, "frames")

SEQUENCE_DIR = max(image_dirs, key=lambda d: sum(1 for _ in d.iterdir()))
print("\nusing sequence:", SEQUENCE_DIR.relative_to(extract_dir))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

all_frames = sorted(SEQUENCE_DIR.glob("*.jpg")) or sorted(SEQUENCE_DIR.glob("*.png"))
print(len(all_frames), "frames available in this sequence")

sample_idx = np.linspace(0, len(all_frames) - 1, 5).astype(int)
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, idx in zip(axes, sample_idx, strict=True):
    ax.imshow(Image.open(all_frames[idx]))
    ax.set_title(f"frame {idx}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Curate an evenly-spaced subset

The full sequence is a turntable capture with far more frames than a
`complete`-pairing run needs. Evenly subsampling to `NUM_FRAMES` frames gives
uniform 360-degree coverage; leaving `ReconstructionJobBuilder`'s own
highest-motion-score selection to pick from the full set doesn't guarantee
that spread. `collect_input_images` (used below) reads a flat directory
non-recursively, so the curated frames get copied into one clean folder.


In [ ]:
SUBSET_DIR = WORK_DIR / "teddybear_subset"
if SUBSET_DIR.exists():
    shutil.rmtree(SUBSET_DIR)
SUBSET_DIR.mkdir(parents=True)

n = min(NUM_FRAMES, len(all_frames))
indices = sorted(set(np.linspace(0, len(all_frames) - 1, n).astype(int).tolist()))
for i, idx in enumerate(indices):
    src = all_frames[idx]
    shutil.copy(src, SUBSET_DIR / f"frame_{i:03d}{src.suffix}")

n_pairs = len(indices) * (len(indices) - 1) // 2
print(f"curated {len(indices)} evenly-spaced frames (of {len(all_frames)}) into {SUBSET_DIR}")
print(f"complete pairing -> C({len(indices)},2) = {n_pairs} MASt3R pairs")

## 8. Run the pipeline: Phase 1 -> 2 -> 3 -> 5

Settings chosen for the best output this data supports, not the fastest run:

| flag | value | why |
|---|---|---|
| `device` | `cuda` | forced, not `auto` -- fail loudly instead of silently falling back to CPU |
| `image_size` | `512` | native resolution of the pinned checkpoint (`MASt3R_ViTLarge_..._512_...`) |
| `pairing_strategy` | `complete` | exhaustive matching; affordable at 24 frames on a P100, more robust than `swin`'s windowed approximation |
| `tsdf_thresh` | `0.2` | the project's own default for real (non-synthetic) captures -- suppresses flying-pixel streaks from raw per-view depth meshing |
| `seed` | `42` | reproducible run |
| `refinement mode` | `object` | keep only the single largest connected component |
| `smoothing_iters` | `15` | default Taubin smoothing |
| `decimate_target_reduction` | `None` | keep every triangle -- no reason to throw away detail here |
| `articulation` | `biped` | teddy bears have a head/arms/legs; the closest of the four templates |
| `max_skinning_influences` | `4` | standard smooth-skinning influence count |

`run_ingested_pipeline` chains Phase 1 ingestion, Phase 2 MASt3R
reconstruction, Phase 3 refinement and Phase 5 rigging in one call.


In [ ]:
from spatial_ingestion.auto_rigging.models import ArticulationType, AutoRigConfig
from spatial_ingestion.final_pipeline.handoff import run_ingested_pipeline
from spatial_ingestion.reconstruction.cli import DEFAULT_MODEL, collect_input_images
from spatial_ingestion.reconstruction.models import Mast3rRunParams
from spatial_ingestion.refinement import MeshCleaningConfig

image_paths = collect_input_images(SUBSET_DIR)
print(f"{len(image_paths)} input images")

RAW_MESH_PATH = OUT_DIR / "mesh.glb"
REFINED_MESH_PATH = OUT_DIR / "mesh_refined.glb"
RIG_DIR = OUT_DIR / "rig"
RIGGED_GLB_PATH = OUT_DIR / "rigged_mesh.glb"

mast3r_params = Mast3rRunParams(
    model_name=DEFAULT_MODEL,
    device="cuda",
    image_size=512,
    pairing_strategy="complete",
    tsdf_thresh=0.2,
    min_conf_thr=1.5,
    seed=42,
    deterministic=True,
    dry_run=False,
)

refinement_config = MeshCleaningConfig(
    mode="object",
    smoothing_iters=15,
    pass_band=0.1,
    decimate_target_reduction=None,
    verify_watertight=True,
)

rigging_config = AutoRigConfig(
    articulation_type=ArticulationType(ARTICULATION),
    max_skinning_influences=4,
    normalize_mesh=True,
    output_dir=RIG_DIR,
    rigged_output_path=RIGGED_GLB_PATH,
)

t0 = time.time()
result = run_ingested_pipeline(
    image_paths,
    mast3r_params=mast3r_params,
    output_path=RAW_MESH_PATH,
    refinement_config=refinement_config,
    refined_output_path=REFINED_MESH_PATH,
    rigging_config=rigging_config,
    rigged_output_path=RIGGED_GLB_PATH,
    rig_output_dir=RIG_DIR,
)
elapsed = time.time() - t0

print(f"\ndone in {elapsed / 60:.1f} minutes")
print("raw mesh    :", result.raw_mesh_path)
print("refined mesh:", result.refined_mesh_path)
print("rigged glb  :", result.rigged_mesh_path)

## 9. All results -- full diagnostic dump

Everything the pipeline itself recorded: Phase 2's `run_manifest.json`
(model, device, pairing, per-stage timings, TSDF fallback flag), Phase 3's
refinement diagnostics (point/triangle counts, watertight status, warnings),
and Phase 5's skeleton/skinning summary.


In [ ]:
print("=" * 70)
print("PHASE 2 -- MASt3R reconstruction (run_manifest.json)")
print("=" * 70)
run_manifest = json.loads(Path(result.reconstruction_manifest_path).read_text(encoding="utf-8"))
print(json.dumps(run_manifest, indent=2, default=str))

print()
print("=" * 70)
print("PHASE 3 -- mesh refinement diagnostics")
print("=" * 70)
print(json.dumps(result.refinement_diagnostics, indent=2, default=str))

print()
print("=" * 70)
print("PHASE 5 -- auto-rigging")
print("=" * 70)
skeleton = json.loads(Path(result.skeleton_path).read_text(encoding="utf-8"))
skinning = json.loads(Path(result.skinning_weights_path).read_text(encoding="utf-8"))
print("articulation type       :", skeleton["articulation_type"])
print("joints  (", len(skeleton["joints"]), "):", [j["name"] for j in skeleton["joints"]])
print("bones   (", len(skeleton["bones"]), "):", [b["name"] for b in skeleton["bones"]])
print("root joint              :", skeleton["root_joint"])
print("skinned vertices        :", len(skinning["weights"]))
print("max influences / vertex :", skinning["max_influences"])
print("rigging warnings        :", result.rigging_warnings)

print()
print("=" * 70)
print("OUTPUT FILE SIZES")
print("=" * 70)
for p in (result.raw_mesh_path, result.refined_mesh_path, result.rigged_mesh_path):
    if p and Path(p).exists():
        print(f"  {Path(p).name:30s} {Path(p).stat().st_size / 1024**2:8.2f} MB")

## 10. Preview -- mesh + skeleton overlay

Reproduces Phase 5's own normalization (`_prepare_mesh`: recenter on the
bounding-box centroid, then scale by `1 / max(extents)`) on the refined mesh
so it lines up exactly with `skeleton.json`'s joint positions, which were fit
in that same normalized space.


In [ ]:
import trimesh
from mpl_toolkits.mplot3d.art3d import Line3DCollection

refined = trimesh.load(str(result.refined_mesh_path), process=False)
if isinstance(refined, trimesh.Scene):
    refined = trimesh.util.concatenate(tuple(refined.geometry.values()))

normalized = refined.copy()
scale = float(max(normalized.extents))
normalized.apply_translation(-normalized.bounding_box.centroid)
normalized.apply_scale(1.0 / scale)

verts = normalized.vertices
try:
    colors = normalized.visual.vertex_colors[:, :3] / 255.0
except Exception:
    colors = None

joint_pos = {j["name"]: np.array(j["position"]) for j in skeleton["joints"]}
bone_segments = [
    (joint_pos[b["parent_joint"]], joint_pos[b["child_joint"]]) for b in skeleton["bones"]
]
joints_arr = np.array(list(joint_pos.values()))

# Metadata says the template convention is Y-up height, X forward/width, Z
# lateral/depth -- reorder to (x, z, y) so matplotlib's default view reads
# with height pointing up.
fig = plt.figure(figsize=(15, 5))
views = [(10, 0, "front"), (10, 90, "side"), (80, 0, "top")]
sample = np.random.choice(len(verts), size=min(20000, len(verts)), replace=False)
for i, (elev, azim, title) in enumerate(views):
    ax = fig.add_subplot(1, 3, i + 1, projection="3d")
    ax.scatter(
        verts[sample, 0],
        verts[sample, 2],
        verts[sample, 1],
        c=colors[sample] if colors is not None else "gray",
        s=1,
        alpha=0.6,
    )
    segs = [[(p[0], p[2], p[1]), (c[0], c[2], c[1])] for p, c in bone_segments]
    ax.add_collection3d(Line3DCollection(segs, colors="red", linewidths=2))
    ax.scatter(joints_arr[:, 0], joints_arr[:, 2], joints_arr[:, 1], c="red", s=30)
    ax.view_init(elev=elev, azim=azim)
    ax.set_title(f"rigged teddybear -- {title}")
    ax.set_box_aspect([1, 1, 1])
    ax.axis("off")
plt.tight_layout()

preview_path = OUT_DIR / "preview_rig.png"
plt.savefig(preview_path, dpi=150)
plt.show()
print("saved", preview_path)

## 11. Package everything for download

`/kaggle/working` persists across the session and is what the Output tab
lists; `/kaggle/temp` (all the `WORK_DIR` scratch above) is wiped. Download
the zip below before ending the session.


In [ ]:
summary = {
    "num_input_frames": len(image_paths),
    "elapsed_minutes": round(elapsed / 60, 2),
    "raw_mesh": str(result.raw_mesh_path),
    "refined_mesh": str(result.refined_mesh_path),
    "rigged_glb": str(result.rigged_mesh_path),
    "skeleton": str(result.skeleton_path),
    "skinning_weights": str(result.skinning_weights_path),
    "rigging_warnings": result.rigging_warnings,
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))

archive_path = shutil.make_archive(
    str(OUT_DIR.parent / "rigging_demo_output"), "zip", root_dir=OUT_DIR
)
print("\npackaged everything into", archive_path)

## Next: Blender

Import `rigged_mesh.glb` via **File > Import > glTF 2.0** -- it carries the
skinned Armature, joints, and vertex colors in one file. Quick recap of the
settings for the best look (full detail in the earlier conversation):

- Object > Shade Auto Smooth (raw photogrammetry meshes come in flat-shaded).
- Confirm a Color Attribute node feeds Base Color on the Principled BSDF
  (usually automatic for vertex-colored glTF); set Metallic to 0 and
  Roughness ~0.6-0.8 since there's no PBR texture here.
- Light with a neutral HDRI world rather than a single lamp.
- Cycles for the final render; Eevee Next for fast viewport iteration.
- Switch Color Management to Standard if AgX washes out the vertex colors.
- Pose Mode: rotate a couple of major bones to sanity-check the skin
  deformation before investing time in a final render -- Phase 5 is an MVP
  and template fitting is orientation-fragile (see `AUTO_RIGGING.md` section
  8), so it's worth eyeballing first.
